<a href="https://colab.research.google.com/github/farahnahh/NorthStar-Urban-Mobility-Analytics/blob/main/notebooks/01_SQL_R_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NorthStar Urban Mobility and Logistics

## SQL within R Analytics

### Objective

This notebook investigates structured operational inefficiencies within NorthStar Urban Mobility and Logistics using SQL queries executed within R.

The analysis focuses on identifying delivery delays, complaint patterns, operational bottlenecks, inefficient hubs, and route-level performance issues across structured datasets.

---

## Dataset Loading and Environment Setup

The following section imports the required analytical libraries and loads the NorthStar operational datasets into the analytical environment for structured SQL within R investigation.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---

## Importing Analytical Libraries

The following libraries support structured operational analytics, SQL execution within R, data manipulation, and performance investigation across NorthStar operational datasets.

In [2]:
# Load R support in Google Colab
%load_ext rpy2.ipython

In [3]:
%%R

install.packages("sqldf")
install.packages("dplyr")
install.packages("ggplot2")
install.packages("DBI")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’

trying URL 'https://cran.rstudio.com/src/contrib/gsubfn_0.7.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/proto_1.0.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RSQLite_2.4.6.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/chron_2.3-62.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/sqldf_0.4-12.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpNWKNNL/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/dplyr_1.2.1.tar.gz'
Content type 'application/x-gzip' length 923509 bytes (901 KB)
downloaded 901 KB


The downloaded source packages are in
	‘/tmp/RtmpNWKNNL/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rs

In [4]:
%%R

library(sqldf)
library(dplyr)
library(ggplot2)
library(DBI)

Loading required package: gsubfn
Loading required package: proto
Loading required package: RSQLite

Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

Use suppressPackageStartupMessages() to eliminate package startup
messages
In addition: Warning message:
no DISPLAY variable so Tk is not available 


---

## Loading NorthStar Operational Datasets

The following section imports the structured operational datasets required for SQL-based analytical investigation within the NorthStar operational intelligence environment.

In [5]:
%%R

# Define dataset path
base_path <- "/content/drive/MyDrive/NorthStar_Datasets/raw_data/"

# Load operational datasets
deliveries <- read.csv(paste0(base_path, "deliveries.csv"))
complaints <- read.csv(paste0(base_path, "complaints.csv"))
incidents <- read.csv(paste0(base_path, "incidents.csv"))
orders <- read.csv(paste0(base_path, "orders.csv"))
drivers <- read.csv(paste0(base_path, "drivers.csv"))
hubs <- read.csv(paste0(base_path, "hubs.csv"))

# Display dataset dimensions
cat("Deliveries Dataset:", dim(deliveries), "\n")
cat("Complaints Dataset:", dim(complaints), "\n")
cat("Incidents Dataset:", dim(incidents), "\n")
cat("Orders Dataset:", dim(orders), "\n")
cat("Drivers Dataset:", dim(drivers), "\n")
cat("Hubs Dataset:", dim(hubs), "\n")

Deliveries Dataset: 950 13 
Complaints Dataset: 320 10 
Incidents Dataset: 280 7 
Orders Dataset: 1250 11 
Drivers Dataset: 170 8 
Hubs Dataset: 8 5 


---

## Delivery Performance Investigation

The following analysis investigates operational delivery outcomes to identify the distribution of successful, delayed, and failed deliveries across the NorthStar logistics network.

In [6]:
%%R

delivery_performance <- sqldf("
    SELECT
        delivery_status,
        COUNT(*) AS total_deliveries,
        ROUND(
            (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM deliveries)),
            2
        ) AS percentage_share
    FROM deliveries
    GROUP BY delivery_status
    ORDER BY total_deliveries DESC
")

print(delivery_performance)

  delivery_status total_deliveries percentage_share
1          OnTime              616            64.84
2         Delayed              202            21.26
3          Failed              132            13.89


---

## Customer Complaint Pattern Investigation

The following analysis investigates customer complaint categories to identify the most common operational service failures affecting customer experience across the NorthStar platform.

In [7]:
%%R

complaint_patterns <- sqldf("
    SELECT
        complaint_type,
        COUNT(*) AS complaint_count,
        ROUND(
            (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM complaints)),
            2
        ) AS percentage_share
    FROM complaints
    GROUP BY complaint_type
    ORDER BY complaint_count DESC
")

print(complaint_patterns)

     complaint_type complaint_count percentage_share
1             Delay             101            31.56
2      MissedPickup              64            20.00
3          AppIssue              53            16.56
4   DriverBehaviour              51            15.94
5 SupportExperience              20             6.25
6           Billing              16             5.00
7            Damage              15             4.69


---

## Operational Hub Delay Investigation

The following analysis investigates delivery delays across operational hubs to identify high-risk logistics locations contributing to operational inefficiencies within the NorthStar network.

In [8]:
%%R

hub_delay_analysis <- sqldf("
    SELECT
        hub_id,
        COUNT(*) AS delayed_deliveries
    FROM deliveries
    WHERE delivery_status = 'Delayed'
    GROUP BY hub_id
    ORDER BY delayed_deliveries DESC
")

print(hub_delay_analysis)

  hub_id delayed_deliveries
1    H04                 28
2    H06                 27
3    H02                 26
4    H01                 26
5    H07                 25
6    H05                 25
7    H03                 23
8    H08                 22


---

## Operational Incident Severity Investigation

The following analysis investigates operational incident severity levels to identify the distribution of disruption risk across NorthStar delivery operations.

In [9]:
%%R

incident_severity_analysis <- sqldf("
    SELECT
        severity,
        COUNT(*) AS incident_count,
        ROUND(
            (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM incidents)),
            2
        ) AS percentage_share
    FROM incidents
    GROUP BY severity
    ORDER BY incident_count DESC
")

print(incident_severity_analysis)

  severity incident_count percentage_share
1   Medium            106            37.86
2      Low             79            28.21
3     High             68            24.29
4 Critical             27             9.64


---

## Driver Delivery Failure Investigation

The following analysis investigates delivery failure distribution across drivers to identify workforce-related operational inefficiencies contributing to failed logistics execution.

In [10]:
%%R

driver_failure_analysis <- sqldf("
    SELECT
        driver_id,
        COUNT(*) AS failed_deliveries
    FROM deliveries
    WHERE delivery_status = 'Failed'
    GROUP BY driver_id
    ORDER BY failed_deliveries DESC
    LIMIT 10
")

print(driver_failure_analysis)

   driver_id failed_deliveries
1       D133                 4
2       D104                 4
3       D024                 4
4       D131                 3
5       D108                 3
6       D092                 3
7       D083                 3
8       D055                 3
9       D010                 3
10      D004                 3


---

## Route Distance Efficiency Investigation

The following analysis investigates average route distance across delivery outcomes to evaluate whether route complexity and travel distance contribute to operational delivery inefficiencies within the NorthStar network.

In [11]:
%%R

route_distance_analysis <- sqldf("
    SELECT
        delivery_status,
        ROUND(AVG(route_distance_km), 2) AS avg_route_distance
    FROM deliveries
    GROUP BY delivery_status
")

print(route_distance_analysis)

  delivery_status avg_route_distance
1         Delayed              14.67
2          Failed              13.37
3          OnTime              13.78


---

## Delivery Cost Efficiency Investigation

The following analysis investigates operational delivery costs across delivery outcomes to evaluate whether failed and delayed deliveries contribute to increased logistics expenditure within the NorthStar network.

In [12]:
%%R

cost_efficiency_analysis <- sqldf("
    SELECT
        delivery_status,
        ROUND(AVG(fuel_or_charge_cost), 2) AS avg_operational_cost
    FROM deliveries
    GROUP BY delivery_status
")

print(cost_efficiency_analysis)

  delivery_status avg_operational_cost
1         Delayed                13.14
2          Failed                13.15
3          OnTime                12.68


---

## Integrated Delivery and Customer Impact Investigation

The following analysis integrates delivery outcomes with customer satisfaction metrics to evaluate how operational delivery performance influences customer experience across the NorthStar logistics environment.

In [13]:
%%R

integrated_operational_analysis <- sqldf("
    SELECT
        d.delivery_status,
        COUNT(DISTINCT d.delivery_id) AS total_deliveries,
        COUNT(DISTINCT c.complaint_id) AS total_complaints,
        ROUND(
            COUNT(DISTINCT c.complaint_id) * 100.0 /
            COUNT(DISTINCT d.delivery_id),
            2
        ) AS complaint_ratio
    FROM deliveries d
    LEFT JOIN complaints c
        ON d.order_id = c.order_id
    GROUP BY d.delivery_status
    ORDER BY complaint_ratio DESC
")

print(integrated_operational_analysis)

  delivery_status total_deliveries total_complaints complaint_ratio
1          Failed              132               35           26.52
2          OnTime              616              149           24.19
3         Delayed              202               48           23.76


---

# SQL within R Analytical Summary

The SQL within R investigation identified significant operational inefficiencies across the NorthStar Urban Mobility and Logistics environment.

The analysis revealed substantial delayed and failed delivery volumes, indicating inconsistent operational execution across logistics operations. Customer complaint analysis demonstrated that delays, missed pickups, and platform-related issues represented major customer dissatisfaction drivers affecting service experience.

Operational hub investigations identified uneven delay distribution across logistics locations, suggesting the presence of concentrated operational bottlenecks and coordination inefficiencies within specific hubs. Incident severity analysis further demonstrated elevated operational disruption exposure, including high and critical severity operational incidents affecting logistics stability.

Workforce investigation identified delivery failure concentration among specific drivers, indicating possible workforce coordination inconsistencies and operational performance variability. Route distance analysis suggested that delivery inefficiencies may be associated with increased route complexity and travel distance, highlighting potential routing optimisation weaknesses.

Operational cost analysis additionally demonstrated that failed and delayed deliveries contribute to increased logistics expenditure, revealing direct financial consequences associated with operational inefficiency.

Finally, integrated SQL analysis confirmed that delivery failures are strongly associated with elevated customer complaint ratios, demonstrating a direct relationship between operational disruption and customer dissatisfaction.

Overall, the SQL within R investigation revealed that fragmented operational coordination, inefficient routing behaviour, workforce inconsistencies, and operational bottlenecks collectively contribute to reduced operational efficiency and weakened customer experience across the NorthStar environment.